<a href="https://colab.research.google.com/github/megagnom111/kursovaya3/blob/main/KursovayaAAA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Курсовая работа Алексеевой А. А. 318

# Анализ периодов булевых функций по алгоритму A1

## Курсовая работа

**Студент:** Алексеева Алёна Алексеевна

**Группа:** 318

**Научный руководитель:** Селезнёва С.Н.

**Дата:** 2026

---

## Постановка задачи

Нам дан **pi-блок (подстановка)** шифра. Из этого pi-блока мы получаем
8 булевых функций (по числу выходных бит).

**Цель работы:** определить, является ли заданный шифр **уязвимым** для криптоанализа
на основе линейной структуры.

**Метод:** для каждой из 8 функций находим **базис пространства периодов**
с помощью алгоритма A1. Затем ищем **общие периоды** для всех 8 функций.

---

## Что такое период булевой функции?

**Определение:** Вектор $a = (a_1, ..., a_n) \in \{0,1\}^n$ называется **периодом**
функции $f(x_1, ..., x_n)$, если для всех $x \in \{0,1\}^n$ выполняется:

$$
f(x_1 \oplus a_1, ..., x_n \oplus a_n) = f(x_1, ..., x_n)
$$

где $\oplus$ — сложение по модулю 2 (XOR).

**Пример:** Для функции $f(x_1, x_2) = x_1$ вектор $(0,1)$ является периодом,
так как $f(x_1, x_2 \oplus 1) = x_1 = f(x_1, x_2)$.

---

## Криптографическое значение

Наличие ненулевых периодов у булевых функций, используемых в шифрах,
создаёт **линейную структуру**, которая может быть использована для
криптоанализа. Чем больше периодов — тем уязвимее шифр.

---
## Связь с уязвимостью шифра

**Ключевая идея:** Если у всех 8 функций шифра **есть общий ненулевой период**,
то существует сдвиг ключа/сообщения, который **не меняет выход шифра**.
Это делает шифр уязвимым для атак.

---

## Структура работы

1. Построение таблицы истинности и получение 8 булевых функций из pi-блока
2. Построение полиномов Жегалкина для каждой функции
3. Применение алгоритма A1 для поиска периодов
4. Нахождение общих периодов для всех 8 функций
5. Вывод об уязвимости шифра

---

# Часть 1. Установка и импорт библиотек, конфигурация констант.

In [1]:
!pip install galois numpy matplotlib -q

In [2]:
import galois       # для операций над полем GF(2) и метода Гаусса
import numpy as np  # для работы с массивами
from math import log
from typing import Optional
# import matplotlib.pyplot as plt

print("Библиотеки загружены!")

Библиотеки загружены!


In [3]:
# количество функций = количество бит в числах
FUNC_COUNT = None
# длина вектора значений функций = количество чисел
POLINOM_LEN = None

# Часть 2. Построение полиномов Жегалкина для зашифрованных функций.

## По шифру строим таблицу истинности, где получаем определённое количество функций. Для каждой функции строим полином Жегалкина и вектор коэффицентов.

## 1. Построение таблицы истинности

Функция **create_truth_table** берёт перестановку (список из 256 чисел) и раскладывает каждое число на биты.
Получается 8 векторов — по одному на каждый бит выхода.

In [4]:
def create_truth_table(permutation: list[int]) -> list[list[int]]:
    """
    Args:
        permutation: список целых чисел длиной POLINOM_LEN,
                    значения от 0 до POLINOM_LEN
                    ( список шифрования )

    Return:
        список из FUNC_COUNT списков, каждый содержит POLINOM_LEN бит
        ( список векторов значений функций )
        -> list[list[int]]

    """

    columns = [[] for _ in range(FUNC_COUNT)]

    for x in range(POLINOM_LEN):
        y = permutation[x]
        binary = format(y, f'0{FUNC_COUNT}b')

        for bit_pos in range(FUNC_COUNT):
            columns[bit_pos].append(int(binary[bit_pos]))

    return columns

Пример работы данной функции на перестановке меньшего размера:

**[4, 3, 5, 1, 2, 6, 7, 0]**

Количество функций: 3

Длина векторов: 8




In [5]:
print("=" * 60)
print("ПРИМЕР: Получения таблицы истинности и векторов функций из S-блока")
print("=" * 60)

example_sbox = [4, 3, 5, 1, 2, 6, 7, 0]
FUNC_COUNT = 3
POLINOM_LEN = 8
print(f"Перестановка: {example_sbox}")
print(f"Количество функций: {FUNC_COUNT}")
print(f"Количество входов: {POLINOM_LEN}")
print()

columns = create_truth_table(example_sbox)

print("Таблица истинности:")
for i in range(POLINOM_LEN):
    print(f'{i:3}  | ', end='')
    for col in range(FUNC_COUNT):
        print(f'{columns[col][i]} ', end='')
    print("\n")

print()
print("Векторы значений функций:")
for i, col in enumerate(columns):
    print(f"  F{i+1}: {col}")


ПРИМЕР: Получения таблицы истинности и векторов функций из S-блока
Перестановка: [4, 3, 5, 1, 2, 6, 7, 0]
Количество функций: 3
Количество входов: 8

Таблица истинности:
  0  | 1 0 0 

  1  | 0 1 1 

  2  | 1 0 1 

  3  | 0 0 1 

  4  | 0 1 0 

  5  | 1 1 0 

  6  | 1 1 1 

  7  | 0 0 0 


Векторы значений функций:
  F1: [1, 0, 1, 0, 0, 1, 1, 0]
  F2: [0, 1, 0, 0, 1, 1, 1, 0]
  F3: [0, 1, 1, 1, 0, 0, 1, 0]


## 2. Построение полиномов Жегалкина для каждой функции

Функция **fast_transform** строит по вектору значений функции вектор коэффицентов соответствующего полинома Жегалкина. Для построения полиномов используется быстрый алгоритм.

За основу взят алгоритм, описанный в лекциях Алексеева В.Б.

In [6]:
def fast_transform(vector: list[int]) -> list[int]:
    """
    Реализация быстрого алгоритма построения вектора полинома Жегалкина
    Args:
        vector: бинарный вектор значений функции длины POLINOM_LEN
    Result:
        бинарный вектор полинома Жегалкина длины POLINOM_LEN
    """
    n = len(vector)
    a = vector.copy()

    step = 1
    while step < n:
        for i in range(0, n, step * 2):
            for j in range(i, i + step, 1):
                a[j + step] ^= a[j]
        step *= 2

    return a

Реализация вспомогательных функций для форматирования вектора коэффицентов в читаемый вид полинома.

Функция **format_zhegalkin_term** вспомогательная, форматирует терм полинома в вид x1\*x2*..*xn

In [7]:
def format_zhegalkin_term(coeff_index: int) -> str:
    """
    Функция для форматирования терм полинома Жегалкина
    Args:
        coeff_index: целое число от 0 до POLINOM_LEN
    Result:
        строка - представление терма, например, "x1*x3" или "1"
    """
    if coeff_index == 0:
        return "1"

    bin_coeff = format(coeff_index, f'0{FUNC_COUNT}b')
    term = []

    for i, bit in enumerate(bin_coeff):
        if bit == "1":
            term.append(f"x{i + 1}")

    return "*".join(term) if term else "1"

Функция **zhegalkin_polinom** преобразует вектор коэффицентов полинома функции в читаемый вид

In [8]:
def zhegalkin_polinom(coeffs: list[int], func_index = 0) -> str:
    non_zero_terms = []
    for i, coeff in enumerate(coeffs):
        if coeff == 1:
            term = format_zhegalkin_term(i)
            non_zero_terms.append(term)

    if not non_zero_terms:
        return f"F{func_index + 1} = 0"
    else:
        return f"F{func_index + 1} = {" ⊕ ".join(non_zero_terms)}"

Пример построения полинома Жегалкина на одной из функций полученных в предыдущем примере.

In [9]:
print("=" * 60)
print("ПРИМЕР: Построение полинома Жегалкина")
print("=" * 60)

# Функция F1 из предыдущего примера
values_f1 = [1, 0, 1, 0, 0, 1, 1, 0]
print(f"Вектор значений F1: {values_f1}")

coeffs_f1 = fast_transform(values_f1)
print(f"Вектор коэффициентов: {coeffs_f1}")

polynomial = zhegalkin_polinom(coeffs_f1)
print(f"Полином Жегалкина: {polynomial}")

print("\nПроверка (ручной расчёт):")
print("  Полином должен быть: 1 ⊕ x3 ⊕ x1 ⊕ x1x2")

ПРИМЕР: Построение полинома Жегалкина
Вектор значений F1: [1, 0, 1, 0, 0, 1, 1, 0]
Вектор коэффициентов: [1, 1, 0, 0, 1, 0, 1, 0]
Полином Жегалкина: F1 = 1 ⊕ x3 ⊕ x1 ⊕ x1*x2

Проверка (ручной расчёт):
  Полином должен быть: 1 ⊕ x3 ⊕ x1 ⊕ x1x2


# Часть 3. Реализация алгоритма *А1*.
## Идея алгоритма:
Период функции f(x) — это такой вектор a, что f(x+a) = f(x) для всех x.
Алгоритм A1 строит систему линейных уравнений, решениями которой являются все периоды функции f.

## Как это работает:
1. Находим самый "старший" моном в полиноме (с максимальным весом)
2. Строим специальный полином g, который "покрывает" этот моном
3. Вычитаем g из f, получая полином меньшей степени
4. Из g получаем линейные уравнения, которым должны удовлетворять периоды
5. Повторяем, пока степень полинома не станет 0
6. Объединяем все уравнения — получаем систему, задающую все периоды


##Вспомогательные функции

Функция **multiply_polinoms** для умножения полиномов в алгебре Жегалкина.

Перемножает два полинома, учитывая правило x² = x.

*Как работает?*

Для каждой пары мономов из p1 и p2:

- Новый моном = XOR масок (симметрическая разность)
- Коэффициент = AND коэффициентов (умножение в GF(2))
- Складываем по модулю 2 (XOR) одинаковые мономы

In [10]:
def multiply_polinoms(p1: dict, p2: dict) -> dict:
    """
    Args:
        p1, p2: словари {маска: коэффициент} для двух полиномов

    Result:
        словарь {маска: коэффициент} для произведения
    """
    res = {}
    for m1, c1 in p1.items():
        if not c1:
            continue
        for m2, c2 in p2.items():
            if not c2:
                continue
            m = m1 ^ m2  # XOR — объединение переменных с удалением дублей
            res[m] = res.get(m, 0) ^ (c1 & c2)  # сложение по модулю 2
            if res[m] == 0:
                del res[m]
    return res

Функция **c_value** для вычисления коэффицента при замене переменной.

Вычисляет c_f(s - e_i + e_j) — коэффициент при мономе, полученном из монома s заменой переменной i на j.

При построении g_k мы смотрим, какие переменные j из o(s)
"связаны" с переменной i через замену. Если коэффициент
при таком заменённом мономе равен 1, то j входит в линейную форму.

In [11]:
def c_value(coeffs: list[int], s_mask: int, i: int, j: int) -> int:
    """
    Args:
        coeffs: вектор коэффициентов исходного полинома
        s_mask: маска монома s (исходный моном)
        i: номер заменяемой переменной (1-индексация)
        j: номер переменной, на которую заменяем (1-индексация)

    Result:
        0 или 1 — коэффициент при заменённом мономе
    """
    n = FUNC_COUNT

    # Проверяем условие: i должен быть в мономе s, j — не должен
    if ((s_mask >> (n - i)) & 1) == 0 or ((s_mask >> (n - j)) & 1) == 1:
        return 0

    # Строим маску нового монома: удаляем i, добавляем j
    new_mask = s_mask & ~(1 << (n - i)) | (1 << (n - j))
    return coeffs[new_mask] if new_mask < len(coeffs) else 0


## Шаг 2.1 алгоритма A1.

Выбираем среди всех ненулевых коэффициентов полинома тот,
который соответствует моному:
    
1) с наибольшим весом (количеством переменных в мономе)

2) если веса равны — с наибольшим лексикографическим номером

In [12]:
def get_max_weight_mask(coeffs: list[int]) -> int:
    """
    Args:
        coeffs: вектор коэффициентов полинома (длина 2^n)

    Result:
        маска выбранного монома (целое число)
    """
    max_weight = -1  # максимальный вес среди просмотренных
    best_mask = 0    # маска монома с максимальным весом (и номером)

    # Перебираем все возможные мономы (от 0 до 2^n-1)
    for mask, c in enumerate(coeffs):
        if c == 0:
            continue  # коэффициент = 0 — монома нет
        w = bin(mask).count('1')  # вес = количество переменных в мономе
        # Сравниваем: сначала по весу, потом по лексикографическому номеру
        if w > max_weight or (w == max_weight and mask > best_mask):
            max_weight = w
            best_mask = mask
    return best_mask


In [13]:
coeffs = [1, 0, 0, 1, 1, 1, 0, 0]
print(get_max_weight_mask(coeffs))

5


##Шаг 2.2 алгоритма A1.

Строим полином g_k = ∏_{i∈i(s)} (x_i + Σ_{j∈o(s)} c·x_j)

*Зачем это нужно:*
        
g_k — это полином, который содержит моном x^{s_k} (старший моном) и другие мономы меньшего веса. При вычитании g_k из f_k
старший моном уничтожается, и степень полинома понижается.

*Как работает:*

1. Находим i(s) — переменные, входящие в моном s
2. Находим o(s) — переменные, НЕ входящие в моном s
3. Для каждого i ∈ i(s) строим линейную форму:
               x_i + Σ_{j∈o(s)} c_f(s - e_i + e_j)·x_j
4. Перемножаем все эти линейные формы

In [14]:
def build_gk(coeffs: list[int], s_mask: int) -> dict:
    """
    Args:
        coeffs: вектор коэффициентов текущего полинома f_k
        s_mask: маска выбранного монома s_k

    Result:
        словарь {маска: коэффициент} для полинома g_k
    """
    n = FUNC_COUNT

    i_list = []
    o_list = []
    for i in range(n):
        if (s_mask >> (n-i-1)) & 1:
            i_list.append(i+1)
        else:
            o_list.append(i+1)

    # Начинаем с полинома-единицы (1)
    result = {0: 1}

    # Для каждой переменной i из i(s) перемножаем скобки
    for i in i_list:
        # Строим линейную форму для текущего i
        linear = {1 << (n-i): 1}  # начинаем с x_i

        # Добавляем слагаемые c·x_j для всех j из o(s)
        for j in o_list:
            if c_value(coeffs, s_mask, i, j):
                m = 1 << (n - j)
                linear[m] = linear.get(m, 0) ^ 1  # XOR для сложения
                if linear[m] == 0:
                    del linear[m]

        # Умножаем накопленный результат на новую линейную форму
        result = multiply_polinoms(result, linear)

    return result

## Реализация полного алгоритма А1.

*Что делает:*

Строит систему линейных уравнений, решениями которой являются
все периоды булевой функции f, заданной полиномом Жегалкина.
Решения этой системы — все периоды исходной функции.

*Как работает (цикл, пока степень f_k >= 1):*

**Шаг 2.1:** Выбрать s_k — моном максимального веса

**Шаг 2.2:** Построить g_k по формуле из статьи

**Шаг 2.3:** h_k = f_k - g_k (вычитание в GF(2))

**Шаг 2.4:** Добавить уравнения T_k из g_k в общую систему

**Шаг 2.5:** f_{k+1} = h_k, повторить



In [15]:
def algorithm_A1(coeffs: list[int]) -> list[list[int]]:
    """
    Алгоритм A1 (основной).

    Args:
        coeffs: вектор коэффициентов полинома Жегалкина (длина 2^n)

    Result:
        список уравнений, каждое уравнение — список [c1, c2, ..., cn],
        означающий: c1·x1 + c2·x2 + ... + cn·xn = 0
    """
    n = FUNC_COUNT
    f_k = coeffs.copy()      # текущий полином (начинаем с исходного)
    equations = []           # накопленная система уравнений T

    # Основной цикл: пока есть мономы степени >= 1
    while True:
        # Проверяем, есть ли в f_k мономы степени >= 1
        degree_ge_1 = False
        for mask, c in enumerate(f_k):
            if c and bin(mask).count('1') >= 1:
                degree_ge_1 = True
                break
        if not degree_ge_1:
            break  # осталась только константа — выход из цикла

        # Шаг 2.1: выбираем s_k
        s_k = get_max_weight_mask(f_k)

        # Находим i(s_k) и o(s_k)
        i_list = []
        o_list = []
        for i in range(n):
            if (s_k >> (n-i-1)) & 1:
                i_list.append(i+1)
            else:
                o_list.append(i+1)

        # Шаг 2.2: строим g_k
        gk_dict = build_gk(f_k, s_k)

        # Шаг 2.3: h_k = f_k - g_k (в GF(2) вычитание = сложение)
        fk_dict = {m: 1 for m, c in enumerate(f_k) if c}  # f_k как словарь
        hk_dict = fk_dict.copy()
        for m, c in gk_dict.items():
            hk_dict[m] = hk_dict.get(m, 0) ^ c  # XOR
            if hk_dict[m] == 0:
                del hk_dict[m]

        # Обновляем f_k для следующей итерации
        f_k = [0] * len(coeffs)
        for m in hk_dict:
            f_k[m] = 1

        # Шаг 2.4: добавляем уравнения T_k в систему T
        # Каждое уравнение: x_i + Σ_{j∈o(s)} c·x_j = 0
        for i in i_list:
            eq = [0] * n
            eq[i - 1] = 1  # коэффициент при x_i
            for j in o_list:
                if c_value(coeffs, s_k, i, j):
                    eq[j - 1] ^= 1  # добавляем x_j
            equations.append(eq)

    return equations


Пример работы алгоритма для функции **f = x₁x₂ ⊕ x₁x₃**

In [16]:
print("=" * 60)
print("ПРИМЕР: Алгоритм A1 на функции с известным периодом")
print("=" * 60)

# Функция f = x₁x₂ ⊕ x₁x₃ (ожидаемый период (1,1,0))
coeffs = [0] * POLINOM_LEN
coeffs[0b110] = 1  # x₁x₂ (маска 6)
coeffs[0b101] = 1  # x₁x₃ (маска 5)

print(f"Полином: {zhegalkin_polinom(coeffs)}")
print("Ожидаемый период: (0,1,1)")

equations = algorithm_A1(coeffs)

print(f"\nПолучено уравнений: {len(equations)}")
print("Система уравнений:")
for i, eq in enumerate(equations):
    terms = [f"x_{j+1}" for j, v in enumerate(eq) if v == 1]
    print(f"  {i+1}. {' + '.join(terms)} = 0")

ПРИМЕР: Алгоритм A1 на функции с известным периодом
Полином: F1 = x1*x3 ⊕ x1*x2
Ожидаемый период: (0,1,1)

Получено уравнений: 2
Система уравнений:
  1. x_1 = 0
  2. x_2 + x_3 = 0


#Часть 4. Нахождение базиса периодов для функции и нахождение общих периодов для нескольких функций

## Нахождение и вывод периодов для одной функции

Функция **find_periods_basis** находит пространство решений (периодов) с помощью библиотеки galois. Для реализации используется метод Гаусса.

In [17]:
def find_periods_basis(coeffs: list[int]) -> Optional[np.ndarray]:
    """
    Args:
        coeffs: вектор коэффициентов полинома Жегалкина (длина 2^n)

    Result:
        - None: все векторы являются периодами (нет уравнений)
        - np.ndarray: матрица, строки которой — базисные векторы
    """
    equations = algorithm_A1(coeffs)

    if not equations:
        return None  # все векторы — периоды

    # Создаём поле GF(2)
    GF2 = galois.GF(2)

    # Преобразуем уравнения в матрицу (строки = уравнения, столбцы = переменные)
    A = GF2(equations)

    # Находим null space (пространство решений A·x = 0)
    null_space = A.null_space()

    return null_space

## Вспомогательная функция для печати периодов одной функции

In [18]:
def print_periods_summary(basis: Optional[np.ndarray], func_name: str):
    """Выводит информацию о периодах функции."""
    if basis is None:
        print(f"{func_name}: ВСЕ ВЕКТОРЫ являются периодами (размерность = {FUNC_COUNT})")
    elif basis.shape[0] == 0 or (basis.shape[0] == 1 and np.all(basis[0] == 0)):
        print(f"{func_name}: только нулевой период")
    else:
        # Убираем нулевой вектор, если он есть
        nonzero_rows = [row for row in basis if not np.all(row == 0)]
        if not nonzero_rows:
            print(f"{func_name}: только нулевой период")
        else:
            print(f"{func_name}: размерность = {len(nonzero_rows)}")
            print(f"  Базис периодов:")
            for row in nonzero_rows:
                print(f"    {row}")


In [20]:
# ПРИМЕР РАБОТЫ
print("=" * 60)
print("ПРИМЕР: Нахождение базиса периодов")
print("=" * 60)
POLINOM_LEN = 8
FUNC_COUNT = 3
coeffs = [0] * POLINOM_LEN
coeffs[0b110] = 1  # x₁x₂ (маска 6)
coeffs[0b101] = 1  # x₁x₃ (маска 5)

print(f"Полином: {zhegalkin_polinom(coeffs)}")
print()

print_periods_summary(find_periods_basis(coeffs), "F1")


ПРИМЕР: Нахождение базиса периодов
Полином: F1 = x1*x3 ⊕ x1*x2

F1: размерность = 1
  Базис периодов:
    [0 1 1]


## Построение общего базиса для нескольких функций

Функция **find_common_periods** находит периоды, общие для всех функций. Строит матрицу из общей системы уравнений и применяет метод Гаусса.



In [21]:
def find_common_periods(all_coeffs: list[list[int]]) -> np.ndarray:
    """
    Args:
        all_coeffs: список векторов коэффициентов для каждой функции

    Result:
        матрица, строки которой — базис общих периодов
    """
    all_equations = []

    for coeffs in all_coeffs:
        equations = algorithm_A1(coeffs)
        all_equations.extend(equations)

    if not all_equations:
        # Нет уравнений → все векторы являются общими периодами
        return None

    GF2 = galois.GF(2)
    A = GF2(all_equations)
    null_space = A.null_space()

    return null_space

Пример работы функции на нескольких функциях.

In [29]:
FUNC_COUNT = 3
POLINOM_LEN = 8

print("=" * 60)
print("ПРИМЕР: Поиск общих периодов для нескольких функций")
print("=" * 60)

# ----- Создаём функции, зависящие только от x₃ -----
print("\n[1] Создаём функции, которые не зависят от x₁ и x₂")

# F1 = x₃
coeffs1 = [0] * POLINOM_LEN
coeffs1[0b001] = 1  # x₃

# F2 = x₃ ⊕ 1
coeffs2 = [0] * POLINOM_LEN
coeffs2[0b001] = 1  # x₃
coeffs2[0b000] = 1  # 1

# F3 = 0 (нулевая функция)
coeffs3 = [0] * POLINOM_LEN

# F4 = x1x2 ⊕ x2x3 ⊕ x1x3
coeffs4 = [0] * POLINOM_LEN
coeffs4[0b011] = 1
coeffs4[0b101] = 1
coeffs4[0b110] = 1
coeffs4[0b111] = 1


print("  F1 = x₃")
print("  F2 = x₃ ⊕ 1")
print("  F3 = 0")
print("  F4 = x1x2 ⊕ x2x3 ⊕ x1x3")

# ----- Строим базисы для каждой функции по отдельности -----
print("\n[2] Базисы периодов для каждой функции:")
print("-" * 40)

def print_basis(coeffs, name):
    basis = find_periods_basis(coeffs)
    print(f"\n  {name}:")
    if basis is None:
        print("    Все векторы являются периодами")
    elif basis.shape[0] == 0 or (basis.shape[0] == 1 and basis[0].tolist() == [0]*3):
        print("    Только нулевой период")
    else:
        nonzero = [row for row in basis if any(row)]
        print(f"    Размерность = {len(nonzero)}")
        print(f"    Базис: {[row.tolist() for row in nonzero]}")

print_basis(coeffs1, "F1 = x₃")
print_basis(coeffs2, "F2 = x₃ ⊕ 1")
print_basis(coeffs3, "F3 = 0")
print_basis(coeffs4, "F4 = x1x2 ⊕ x2x3 ⊕ x1x3")

print("\n[3] Ищем общие периоды для всех трёх функций")
print("-" * 40)

all_coeffs = [coeffs1, coeffs2, coeffs3, coeffs4]
common_basis = find_common_periods(all_coeffs)

print("\n[4] Результат find_common_periods:")
print("-" * 40)

nonzero = [row for row in common_basis if any(row)]
if not nonzero:
    print("  Общих ненулевых периодов нет (только нулевой)")
else:
    print(f"  Размерность общих периодов = {len(nonzero)}")
    print(f"  Базис общих периодов:")
    for row in nonzero:
        print(f"    {row.tolist()}")

    print(f"\n  Все общие периоды (включая нулевой): {2**len(nonzero)}")



ПРИМЕР: Поиск общих периодов для нескольких функций

[1] Создаём функции, которые не зависят от x₁ и x₂
  F1 = x₃
  F2 = x₃ ⊕ 1
  F3 = 0
  F4 = x1x2 ⊕ x2x3 ⊕ x1x3

[2] Базисы периодов для каждой функции:
----------------------------------------

  F1 = x₃:
    Размерность = 2
    Базис: [[1, 0, 0], [0, 1, 0]]

  F2 = x₃ ⊕ 1:
    Размерность = 2
    Базис: [[1, 0, 0], [0, 1, 0]]

  F3 = 0:
    Все векторы являются периодами

  F4 = x1x2 ⊕ x2x3 ⊕ x1x3:
    Только нулевой период

[3] Ищем общие периоды для всех трёх функций
----------------------------------------

[4] Результат find_common_periods:
----------------------------------------
  Общих ненулевых периодов нет (только нулевой)


## Часть 5. Общий алгоритм выполнения задачи, нахождение уязвимостей и реализация на наших данных

Опишем функцию **analyze_permutation** общего анализа для любых входных перестановок.

In [55]:
def analyze_permutation(permutation: list[int]):
    """
    Полный анализ перестановки (pi-блока).
    """
    print("=" * 80)
    print("АНАЛИЗ ПЕРИОДОВ БУЛЕВЫХ ФУНКЦИЙ НА ОСНОВЕ ВХОДНОЙ ПЕРЕСТАНОВКИ")
    print("=" * 80)
    print(f"Число переменных: n = {FUNC_COUNT}")
    print(f"Размер таблицы: 2^{FUNC_COUNT} = {POLINOM_LEN}")
    print()


    # Шаг 1: получаем векторы значений
    columns = create_truth_table(permutation)

    # Шаг 2: для каждой функции находим коэффициенты и периоды
    all_coeffs = []

    for i in range(FUNC_COUNT):
        print(f"\n--- Функция F{i+1} = {columns[i]} ---")
        # Вектор коэффициентов полинома
        coeffs = fast_transform(columns[i])
        all_coeffs.append(coeffs)

        # Печатаем полином
        print(f"Полином Жегалкина: {zhegalkin_polinom(coeffs)}")

        print(f"  Уравнения (алгоритм A1):")
        eqs = algorithm_A1(coeffs)
        for eq in eqs[:5]:  # показываем не более 5
            terms = [f"x{j+1}" for j, val in enumerate(eq) if val == 1]
            print(f"    {' + '.join(terms)} = 0")
        if len(eqs) > 5:
            print(f"    ... и ещё {len(eqs)-5} уравнений")

        # Находим периоды
        basis = find_periods_basis(coeffs)
        print_periods_summary(basis, f"F{i+1}")

    print(f"\n--- Общий базис системы функций ---")
    common_basis = find_common_periods(all_coeffs)

    nonzero = [row for row in common_basis if any(row)]
    if not nonzero:
        print("  Общих ненулевых периодов нет (только нулевой)")
        print(f"\n  Все общие периоды (включая нулевой): {2**len(nonzero)}")
        print(f"  Шифр УСТОЙЧИВ к атакам на основе линейной структуры!")
    else:
        print(f"  Размерность общих периодов = {len(nonzero)}")
        print(f"  Базис общих периодов:")
        for row in nonzero:
            print(f"    {row.tolist()}")
        print(f"\n  Все общие периоды (включая нулевой): {2**len(nonzero)}")
        print(f"\n  Существует сдвиг, который не меняет НИ ОДНУ из {FUNC_COUNT} функций")
        print(f"  Шифр УЯЗВИМ для атак на основе линейной структуры!")



In [56]:
FUNC_COUNT = 3
POLINOM_LEN = 8
permutation = [4, 3, 5, 1, 2, 6, 7, 0]
analyze_permutation(permutation)

АНАЛИЗ ПЕРИОДОВ БУЛЕВЫХ ФУНКЦИЙ НА ОСНОВЕ ВХОДНОЙ ПЕРЕСТАНОВКИ
Число переменных: n = 3
Размер таблицы: 2^3 = 8


--- Функция F1 = [1, 0, 1, 0, 0, 1, 1, 0] ---
Полином Жегалкина: F1 = 1 ⊕ x3 ⊕ x1 ⊕ x1*x2
  Уравнения (алгоритм A1):
    x1 = 0
    x2 = 0
    x1 + x3 = 0
F1: только нулевой период

--- Функция F2 = [0, 1, 0, 0, 1, 1, 1, 0] ---
Полином Жегалкина: F1 = x3 ⊕ x2*x3 ⊕ x1 ⊕ x1*x3
  Уравнения (алгоритм A1):
    x1 + x2 = 0
    x3 = 0
    x1 + x3 = 0
F2: только нулевой период

--- Функция F3 = [0, 1, 1, 1, 0, 0, 1, 0] ---
Полином Жегалкина: F1 = x3 ⊕ x2 ⊕ x2*x3 ⊕ x1*x3
  Уравнения (алгоритм A1):
    x1 + x2 = 0
    x3 = 0
    x2 + x3 = 0
F3: только нулевой период

--- Общий базис системы функций ---
  Общих ненулевых периодов нет (только нулевой)

  Все общие периоды (включая нулевой): 1
  Шифр УСТОЙЧИВ к атакам на основе линейной структуры!


In [57]:
FUNC_COUNT = 8
POLINOM_LEN = 256
permutation = [252, 238, 221, 17, 207, 110, 49, 22, 251, 196, 250, 218, 35, 197, 4, 77, 233, 119, 240, 219, 147, 46, 153, 186, 23, 54, 241, 187, 20, 205, 95, 193, 249, 24, 101, 90, 226, 92, 239, 33, 129, 28, 60, 66, 139, 1, 142, 79, 5, 132, 2, 174, 227, 106, 143, 160, 6, 11, 237, 152, 127, 212, 211, 31, 235, 52, 44, 81, 234, 200, 72, 171, 242, 42, 104, 162, 253, 58, 206, 204, 181, 112, 14, 86, 8, 12, 118, 18, 191, 114, 19, 71, 156,
183, 93, 135, 21, 161, 150, 41, 16, 123, 154, 199, 243, 145, 120, 111, 157, 158, 178, 177, 50, 117, 25, 61, 255, 53, 138, 126, 109, 84, 198, 128, 195, 189, 13, 87, 223, 245, 36, 169, 62, 168, 67, 201, 215, 121, 214, 246, 124, 34, 185, 3, 224, 15, 236, 222, 122, 148, 176, 188, 220, 232, 40, 80, 78, 51, 10, 74, 167, 151, 96, 115, 30, 0, 98, 68, 26, 184, 56, 130, 100, 159, 38, 65, 173, 69, 70, 146, 39, 94, 85, 47, 140, 163, 165, 125, 105, 213, 149, 59, 7, 88, 179, 64, 134, 172, 29, 247, 48, 55, 107, 228, 136, 217, 231, 137, 225, 27, 131, 73, 76, 63, 248, 254, 141, 83, 170, 144, 202, 216, 133, 97, 32, 113, 103, 164, 45, 43, 9, 91, 203, 155, 37, 208, 190, 229, 108, 82, 89, 166, 116, 210, 230, 244, 180, 192, 209, 102, 175, 194, 57, 75, 99, 182
]
analyze_permutation(permutation)

АНАЛИЗ ПЕРИОДОВ БУЛЕВЫХ ФУНКЦИЙ НА ОСНОВЕ ВХОДНОЙ ПЕРЕСТАНОВКИ
Число переменных: n = 8
Размер таблицы: 2^8 = 256


--- Функция F1 = [1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1] ---
Полином Жегалкина: F1 = 1 ⊕ x7*x8 ⊕ x6*x8 ⊕ x6*x7 ⊕ x5*x7*x8 ⊕ x5*x6 ⊕ x5*x6*x7 ⊕ x5*x6*x7*x8 ⊕